In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.functions import * 

#### Physical Join Strategies

* Shuffle Hash join:
    * Both datasets are shuffled
    * Smaller dataset is hashed
    * Hased data join with big dataset

* Sort - Merge join:
    * Both datsets are shuffled
    * The keys that used to join tables are sorted properly
    * Then the datasets will be merged
    * Used for 2 big datasets

* Broadcast join
    * No shuffle
    * Small datasets are broadcasted (copied) to all the executors
    * Joining takes place once the dataset is broadcased to all executors

In [2]:
spark = SparkSession.builder \
    .appName("optimizeJoin") \
    .master("local[*]") \
    .config("spark.executor.memory", "512M") \
    .config("spark.executor.cores",4) \
    .config("spark.sql.adaptive.enabled", True) \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/19 19:01:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark

In [4]:
sales_schema = """
    transacted_at string,
    trx_id string,
    retailer_id string,
    description string,
    amount double,
    city_id string
"""

sales = spark.read.format("csv").schema(sales_schema).option("header",True).load("/Users/AnhHuynh/Documents/PySpark/new_sales.csv")

In [5]:
sales.rdd.getNumPartitions()

8

In [6]:
sales.printSchema()

root
 |-- transacted_at: string (nullable = true)
 |-- trx_id: string (nullable = true)
 |-- retailer_id: string (nullable = true)
 |-- description: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- city_id: string (nullable = true)



In [7]:
dept_schema = """

department_id int,
department_name string,
description string,
city string,
state string,
country string
"""

dept = spark.read.format("csv").schema(dept_schema).option("header",True).load("/Users/AnhHuynh/Documents/PySpark/department_data.txt")

In [8]:
emp_schema = """ 
    first_name string,
    last_name string,
    job_title string,
    dob date,
    email string,
    phone string,
    salary float,
    department_id int
"""

emp = spark.read.format("csv").schema(emp_schema).option("header",True).load("/Users/AnhHuynh/Documents/PySpark/employee_records.txt")

In [9]:
city_schema = """ 
    city_id string,
    city string,
    state string,
    state_abv string,
    country string
"""

city = spark.read.format("csv").schema(city_schema).option("header",True).load("/Users/AnhHuynh/Documents/PySpark/cities.txt")

In [10]:
city.rdd.getNumPartitions()

8

In [11]:
city.select("city_id").distinct().count()

2349391

#### Optimize joining Big and Small tables using Broadcast

In [12]:
df_joined = emp.join(broadcast(dept), how="left_outer", on=emp.department_id==dept.department_id).drop(dept.department_id)

df_joined.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [first_name#18, last_name#19, job_title#20, dob#21, email#22, phone#23, salary#24, department_id#25, department_name#13, description#14, city#15, state#16, country#17]
   +- BroadcastHashJoin [department_id#25], [department_id#12], LeftOuter, BuildRight, false
      :- FileScan csv [first_name#18,last_name#19,job_title#20,dob#21,email#22,phone#23,salary#24,department_id#25] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/AnhHuynh/Documents/PySpark/employee_records.txt], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<first_name:string,last_name:string,job_title:string,dob:date,email:string,phone:string,sal...
      +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=114]
         +- Filter isnotnull(department_id#12)
            +- FileScan csv [department_id#12,department_name#13,description#14,city#15,st

In [13]:
df_joined.write.format("noop").mode("overwrite").save()

In [14]:
df_joined.show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------------------+--------------------+--------------------+-----+-------------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|     department_name|         description|                city|state|            country|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------------------+--------------------+--------------------+-----+-------------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8|          Parker PLC|Assimilated multi...|         Barnettside|   AL|   Marshall Islands|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7|         Ward-Gordon|Progressive logis..

#### Optimze Sort-Merge Join on Big and Big tables using Bucketing

🔌 The Golden Rules of Bucketing for Spark to eliminate the shuffle step, both datasets must meet these exact criteria when saved: 
* Use the exact same number of buckets.
* Bucket on the exact same column (city_id).
* The bucket and join column should be the same
* Be saved as persistent managed or unmanaged Spark tables

🌟 The golden rule of bucketing is that each individual bucket file should ideally be between 128 MB and 256 MB in size after being written to storage.

In [15]:
# Write sales data into buckets
sales.write.format("csv").mode("overwrite").bucketBy(4,"city_id").option("header",True).option("path","/Users/AnhHuynh/Documents/PySpark/sales_bucket.csv").saveAsTable("sales_bucket")

In [16]:
city.write.format("csv").mode("overwrite").bucketBy(4,"city_id").option("header",True).option("path","/Users/AnhHuynh/Documents/PySpark/city_bucket.csv").saveAsTable("city_bucket")

In [17]:
spark.sql("show tables in default").show()

+---------+------------+-----------+
|namespace|   tableName|isTemporary|
+---------+------------+-----------+
|  default| city_bucket|      false|
|  default|sales_bucket|      false|
+---------+------------+-----------+



#### Join Sales and City data using Sort-Merge with Bucketing

In [18]:
sales_bucket = spark.read.table("sales_bucket")

In [19]:
city_bucket = spark.read.table("city_bucket")

In [20]:
# Join datasets

df_bucket = sales_bucket.join(city_bucket, on=sales_bucket.city_id==city_bucket.city_id,how="left")

In [ ]:
# Write dataset
df_bucket.write.format("noop").mode("overwrite").save()

26/06/19 19:13:06 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 123807 ms exceeds timeout 120000 ms
26/06/19 19:13:06 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/19 19:13:09 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1363)
	at o

In [ ]:
df_bucket.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [city_id#152], [city_id#153], LeftOuter
   :- Sort [city_id#152 ASC NULLS FIRST], false, 0
   :  +- FileScan csv spark_catalog.default.sales_bucket[transacted_at#147,trx_id#148,retailer_id#149,description#150,amount#151,city_id#152] Batched: false, Bucketed: true, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/AnhHuynh/Documents/PySpark/sales_bucket.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<transacted_at:string,trx_id:string,retailer_id:string,description:string,amount:double,cit..., SelectedBucketsCount: 4 out of 4
   +- Sort [city_id#153 ASC NULLS FIRST], false, 0
      +- Filter isnotnull(city_id#153)
         +- FileScan csv spark_catalog.default.city_bucket[city_id#153,city#154,state#155,state_abv#156,country#157] Batched: false, Bucketed: true, DataFilters: [isnotnull(city_id#153)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/AnhHuynh/

26/06/19 19:15:29 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1363)
	at org.apache.spark.executor.Executor.$anonfun$heartbeater$1(Executor.scala:356)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.scala:18)
	at org.apache.spark.util.Utils$.logUncaughtExceptions(Utils.scala:1941